In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import scipy
import numpy as np
import xskillscore as xs

import dask

import albedo_functions as af

In [ ]:
def dataset_season(dset,season):
    """
    SEASON SELECTION
    """
    if season == 'DJF':
        mon = [12,1,2]
    elif season == 'MAM':
        mon = [3,4,5]
    elif season == 'JJA':
        mon = [6,7,8]
    elif season == 'SON':
        mon = [9,10,11]
    elif season == 'Clim':
        mon = [11,12,1,2,3,4,5,6,7,8,9,10]
        
    dset = dset.sel(time=dset.time.dt.month.isin(mon))
    dset = dset.rolling(time = len(mon)).mean('time')
    dset = dset.sel(time = dset.time.dt.month == mon[-1])
    
    return dset

In [ ]:
def p_cal(ctrl, sens):
    a = ctrl.resample(time='YE').mean().var(dim='time')
    b = sens.resample(time='YE').mean().var(dim='time')

# Add a small constant to avoid division by zero
    epsilon = 1e-8
    a += epsilon
    b += epsilon

    F = a / b
    #df = len(ctrl.resample(time='YE').mean())
    df = ctrl.resample(time='YE').mean().sizes['time']
    p_value = scipy.stats.f.cdf(F, df, df)
    p_value = xr.Dataset({
        'p': xr.DataArray(p_value, dims=('lat', 'lon'),
                          coords={'lon': a['lon'],'lat': a['lat']})
    })
    return p_value

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
BASE_PATH = f'{POST_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
def processing_std(ctrl_path, sens_path, var, season):
    ctrl = xr.open_dataset(ctrl_path, chunks={'time': 12})[var]
    ctrl = xr.where(ctrl.std('time') >= 0.0005, ctrl, np.nan)
    sens = xr.open_dataset(sens_path, chunks={'time': 12})[var]
    sens = xr.where(sens.std('time') >= 0.0005, sens, np.nan)
    
    # Extract seasonal data
    seasonal_sens = dataset_season(sens,season)
    seasonal_ctrl = dataset_season(ctrl,season)
    
    # Compute standard deviation over time (interannual)    
    sens_std = seasonal_sens.std(dim='time')
    ctrl_std = seasonal_ctrl.std(dim='time')
    
    # Compute p-values
    p_value = af.land_seas_mask(p_cal(seasonal_ctrl,seasonal_sens))

    # Convert to Dataset and save
    ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_{var}_{season}_std.nc')
    sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_{var}_{season}_std.nc')
    p_value.to_netcdf(f'{SAVE_PATH}delta_{var}_{season}_std_p.nc')

In [ ]:
%%time
paths = {
    'cvh': {
        'ctrl': BASE_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc',
        'sens': BASE_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc'
    },
    'cvl': {
        'ctrl': BASE_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc',
        'sens': BASE_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc'
    },
}
for var, pair in paths.items():
    for season in ['DJF', 'JJA', 'MAM', 'SON']:
        processing_std(pair['ctrl'], pair['sens'], var, season)